<!--nav--> [🗺 Learning path](README.md) · **43/49** · ◀ [RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) · [Production Hardening](./Production_Hardening_Reliability.ipynb) ▶

# Agent Workloads on the Metal: What Changes When the Caller Is a Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Agent_Workloads_On_The_Metal.ipynb)

Every serving notebook in this repo so far assumed a chat: a person types, the model answers,
the conversation grows slowly, and the expensive part is generating tokens one at a time.

An agent is not that. An agent turn is:

1. re-send **everything** — system prompt, tool definitions, the whole conversation, every
   tool result so far
2. generate a **short** structured output — usually a JSON tool call, a hundred tokens or so
3. run a tool, which takes **seconds** while the sequence sits there holding its KV cache
4. append the result and go back to step 1

Seven properties fall out of that shape. Each one lands on a kernel, and each of those
kernels is in this repository, compiles, and runs here:

| what is different about an agent | what it costs | the kernel |
|---|---|---|
| the same context arrives again and again | re-prefilling what the server already has | [`16_prefix_match.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/16_prefix_match.cu) |
| branches share a long prefix | re-*reading* that prefix N times per step | [`13_prefix_attention.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/13_prefix_attention.cu) |
| every agent is on a different turn | a batch padded to its longest sequence | [`17_ragged_batch.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/17_ragged_batch.cu) |
| output is JSON, at temperature 0 | a grammar mask on nearly every token | [`14_logit_mask.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/14_logit_mask.cu) |
| that output is highly predictable | a speedup most workloads cannot get | [`18_spec_verify.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/18_spec_verify.cu) |
| trajectories fork | duplicating a KV cache per branch | [`15_kv_fork.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/15_kv_fork.cu) |
| runs are replayed, retried and evaluated | logits that depend on the batch | [`19_batch_invariant.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/19_batch_invariant.cu) |

And one that lands on no kernel at all and matters more than most of them: **an agent sits
idle, holding memory, while its tools run.**

Two things are worth saying before any of it. First, the biggest lever here is not a kernel —
it is the decision not to recompute, and the kernel that makes that decision (`16`) is the
cheapest code in the whole serving path. Second, none of these kernels is new. Every one is a
kernel from earlier in this repository pointed at a different caller: `13` and `17` are `06`'s
online-softmax merge over two different partitions, `15` is `06`'s block table used for a
purpose PagedAttention did not have in mind, `18` is `14`'s masked argmax run k+1 times, and
`19` is `02`'s reduction with the split count taken away from the scheduler. That is the real
claim of this notebook: **agent serving is not a new set of kernels, it is a new set of
reasons to reach for the ones you have.**

All seven run here with no GPU — they compile with `g++` against the CPU shim. With a GPU you
also get the timings.

In [1]:
# Setup. On Colab this clones the repo; run it and everything below works.
import os, subprocess, sys, math, json, uuid
from pathlib import Path
from IPython.display import HTML, display

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "kernels" / "Makefile").exists():
            return cand
    return None

URL = "https://github.com/sugeerth/gpu-training-notebooks"
BRANCH = "claude/serving-optimization-notebooks-jyerty"   # until this lands on main

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "kernels" / "Makefile").exists():
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", URL, str(dest)], check=True)
        if not (dest / "kernels" / "Makefile").exists():
            subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=str(dest))
            subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=str(dest))
    REPO = dest
KERNELS = REPO / "kernels"

def sh(cmd, cwd=KERNELS, limit=7000):
    """Run a command and show its output, minus the ##KB## line the eval harness reads."""
    p = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    clean = "\n".join(l for l in (p.stdout or "").splitlines() if not l.startswith("##KB##"))
    out = clean[-limit:]
    if out:
        print(out, end="" if out.endswith("\n") else "\n")
    if p.returncode != 0 and p.stderr:
        print((p.stderr or "")[-2000:], file=sys.stderr)
    return p.returncode

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    display(HTML(f"""
<div id="{div}" style="width:100%;max-width:880px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function go() {{ const DATA = {json.dumps(data)}; const root = d3.select("#{div}"); {js} }}
  if (window.d3) go();
  else {{ const s=document.createElement("script"); s.src="{D3_URL}"; s.onload=go;
          document.head.appendChild(s); }}
}})();
</script>"""))

print("repo :", REPO)
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    print("gpu  :", torch.cuda.get_device_name(0) if HAVE_GPU else "none — correctness only")
except ImportError:
    torch, HAVE_GPU = None, False
    print("gpu  : no torch — correctness only")

repo : /home/user/gpu-training-notebooks


gpu  : none — correctness only


## Part 1 · The prefill wall

A chat turn appends a sentence. An agent turn appends a tool result — often a page of search
results, a file, a stack trace — and then re-sends the whole thing.

So the context grows by a fixed amount every turn, and **every turn prefills the entire
context from scratch** unless something stops it. Summed over an `N`-turn run that is

$$\sum_{i=0}^{N-1} \left(c_0 + i \cdot g\right) = N c_0 + g\frac{N(N-1)}{2}$$

— **quadratic in the number of turns**. With prefix caching, turn `i` prefills only the `g`
new tokens, and the total is linear: `c_0 + (N-1)·g`.

That is the single largest lever in agent serving, and it is worth being precise about what it
is and is not. It is not a kernel optimization. It is the observation that the first `c_i - g`
tokens of turn `i` are byte-identical to turn `i-1`, so their K and V are already in the pool
and never need recomputing. The kernels come in afterwards — for *reading* that prefix
efficiently, which is Part 2.

In [2]:
# The agent-run model. This function is the source of truth: demo/agent-loop.html reproduces
# it in JavaScript and tools/verify_console.py checks the two agree.
def agent_run(turns=20, system_tokens=2000, tool_def_tokens=1500, user_tokens=200,
              response_tokens=150, tool_result_tokens=800, tool_latency_s=2.0,
              prefix_hit_rate=0.95, fanout=1, prefill_tok_per_s=20000.0,
              decode_tok_per_s=60.0, kv_bytes_per_token=131072.0, pool_gb=40.0,
              price_in_per_mtok=3.0, price_out_per_mtok=15.0):
    """Where an agent run's tokens, time, memory and money actually go.

    Everything is exact arithmetic on the shape of the loop. The rates are the planning
    assumptions — measure your own and substitute them.
    """
    base = system_tokens + tool_def_tokens + user_tokens   # context before the first turn
    growth = response_tokens + tool_result_tokens          # added by each completed turn

    # Prefill, three ways.
    #   naive  — re-prefill the whole context every turn: quadratic in `turns`
    #   ideal  — only ever prefill tokens the server has never seen: linear
    #   actual — a cache that misses some fraction of the time, which is the real world
    naive = 0.0
    ideal = 0.0
    actual = 0.0
    contexts = []
    for i in range(turns):
        ctx = base + i * growth
        contexts.append(ctx)
        new = ctx if i == 0 else growth        # tokens this turn has never sent before
        cacheable = ctx - new
        naive += ctx
        ideal += new
        actual += new + (1.0 - prefix_hit_rate) * cacheable

    final_ctx = base + (turns - 1) * growth if turns else base
    decode_tokens = turns * response_tokens * fanout

    prefill_s = actual / prefill_tok_per_s
    decode_s = decode_tokens / decode_tok_per_s
    tool_s = turns * tool_latency_s
    wall_s = prefill_s + decode_s + tool_s

    # The KV a single agent holds at the end of the run, times its branches.
    kv_gb = final_ctx * kv_bytes_per_token * fanout / 1e9
    seats = pool_gb / kv_gb if kv_gb > 0 else float("inf")

    # A slot is "working" only while the GPU is doing something for it. During a tool call it
    # holds its KV and produces nothing.
    busy_s = prefill_s + decode_s
    slot_util = busy_s / wall_s if wall_s > 0 else 0.0

    cost = actual / 1e6 * price_in_per_mtok + decode_tokens / 1e6 * price_out_per_mtok
    cost_naive = naive / 1e6 * price_in_per_mtok + decode_tokens / 1e6 * price_out_per_mtok

    return dict(
        base_tokens=base, growth_per_turn=growth, final_context=final_ctx,
        prefill_naive=naive, prefill_ideal=ideal, prefill_actual=actual,
        decode_tokens=decode_tokens,
        prefill_s=prefill_s, decode_s=decode_s, tool_s=tool_s, wall_s=wall_s,
        kv_gb=kv_gb, concurrent_agents=seats, slot_util=slot_util,
        cost_usd=cost, cost_usd_naive=cost_naive,
        prefill_share=prefill_s / wall_s if wall_s else 0.0,
        decode_share=decode_s / wall_s if wall_s else 0.0,
        tool_share=tool_s / wall_s if wall_s else 0.0,
    )


r = agent_run()
print("A 20-turn agent: 2k system prompt, 1.5k tool definitions, 800-token tool results\n")
print(f"  context at the end        {r['final_context']:>12,.0f} tokens")
print(f"  prefill, no caching       {r['prefill_naive']:>12,.0f} tokens")
print(f"  prefill, perfect caching  {r['prefill_ideal']:>12,.0f} tokens"
      f"   ({r['prefill_naive']/r['prefill_ideal']:.1f}x less)")
print(f"  prefill, 95% hit rate     {r['prefill_actual']:>12,.0f} tokens"
      f"   ({r['prefill_actual']/r['prefill_ideal']:.1f}x the ideal)")
print(f"  decode                    {r['decode_tokens']:>12,.0f} tokens")
print(f"\n  wall clock {r['wall_s']:.0f}s = prefill {r['prefill_s']:.1f}s"
      f" + decode {r['decode_s']:.1f}s + tools {r['tool_s']:.0f}s")
print(f"  cost ${r['cost_usd']:.3f}  (without caching, ${r['cost_usd_naive']:.3f})")
print("""
Note where the 5% of misses lands. A 95% hit rate sounds like a rounding error and costs 1.5x
the ideal prefill, because the 5% it misses is 5% of a *large* number — the whole context —
not 5% of the new tokens. Prefix cache hit rate is one of the few metrics where the difference
between 95% and 99% is worth engineering for.""")

A 20-turn agent: 2k system prompt, 1.5k tool definitions, 800-token tool results

  context at the end              21,750 tokens
  prefill, no caching            254,500 tokens
  prefill, perfect caching        21,750 tokens   (11.7x less)
  prefill, 95% hit rate           33,388 tokens   (1.5x the ideal)
  decode                           3,000 tokens

  wall clock 92s = prefill 1.7s + decode 50.0s + tools 40s
  cost $0.145  (without caching, $0.809)

Note where the 5% of misses lands. A 95% hit rate sounds like a rounding error and costs 1.5x
the ideal prefill, because the 5% it misses is 5% of a *large* number — the whole context —
not 5% of the new tokens. Prefix cache hit rate is one of the few metrics where the difference
between 95% and 99% is worth engineering for.


In [3]:
# The quadratic, made visible. Turns on the x-axis, cumulative prefill tokens on the y.
rows = []
for n in range(1, 61):
    a = agent_run(turns=n, tool_result_tokens=2000)
    rows.append({"n": n, "naive": a["prefill_naive"], "ideal": a["prefill_ideal"],
                 "actual": a["prefill_actual"]})

show_d3(r"""
  const W = 840, H = 330, M = {t: 26, r: 150, b: 44, l: 74};
  const svg = root.append("svg").attr("width", W).attr("height", H);
  const x = d3.scaleLinear().domain([1, d3.max(DATA, d => d.n)]).range([M.l, W - M.r]);
  const y = d3.scaleLinear().domain([0, d3.max(DATA, d => d.naive) * 1.05]).range([H - M.b, M.t]);
  svg.append("g").attr("transform", `translate(0,${H - M.b})`).call(d3.axisBottom(x).ticks(8));
  svg.append("g").attr("transform", `translate(${M.l},0)`)
     .call(d3.axisLeft(y).ticks(6).tickFormat(d => (d / 1e6).toFixed(1) + "M"));
  svg.append("text").attr("x", M.l).attr("y", 14).style("font-size", "12px")
     .style("fill", "#6f8085").text("cumulative prefill tokens");
  svg.append("text").attr("x", (M.l + W - M.r) / 2).attr("y", H - 6).attr("text-anchor", "middle")
     .style("font-size", "12px").style("fill", "#6f8085").text("agent turns");
  const series = [["naive", "#a33131"], ["actual", "#b45210"], ["ideal", "#0a6f78"]];
  const label = {naive: "no caching — quadratic", actual: "95% hit rate",
                 ideal: "perfect caching — linear"};
  series.forEach(([k, c]) => {
    svg.append("path").datum(DATA).attr("fill", "none").attr("stroke", c).attr("stroke-width", 2.4)
       .attr("d", d3.line().x(d => x(d.n)).y(d => y(d[k])));
    const last = DATA[DATA.length - 1];
    svg.append("text").attr("x", x(last.n) + 8).attr("y", y(last[k]) + 4)
       .style("font-size", "12px").style("fill", c).text(label[k]);
  });
""", rows)

last = rows[-1]
print(f"At 60 turns with 2000-token tool results: {last['naive']/1e6:.2f}M tokens prefilled")
print(f"without caching, {last['ideal']/1e3:.0f}k with it — {last['naive']/last['ideal']:.0f}x.")
print("""
The gap is not a constant factor. It grows linearly with the number of turns, because one
curve is quadratic and the other is not. A 5-turn agent barely notices; a 50-turn one is
paying for prefill it already did forty times.""")

At 60 turns with 2000-token tool results: 4.03M tokens prefilled
without caching, 131k with it — 31x.

The gap is not a constant factor. It grows linearly with the number of turns, because one
curve is quadratic and the other is not. A 5-turn agent barely notices; a 50-turn one is
paying for prefill it already did forty times.


### The kernel that decides whether any of this happens

Everything above is arithmetic about a decision. The decision itself is a kernel: given an
arriving context, how many of its leading blocks does the server already hold KV for?

Three things make it harder than a dictionary lookup, and all three are visible in the code.
It is a **prefix** match — attention at position `i` depends on every token before it, so a
block that matches after a gap is worthless. It is **block-granular** — KV lives in pages, so
the match is quantized. And a block's identity has to encode its **whole history**, not its own
tokens, because the same sixteen tokens after a different prefix have different KV. That last
one forces a chained hash, which is serial, and most of the kernel is about what can be made
parallel around it.

In [4]:
sh("make --no-print-directory 16_prefix_match")

=== 16_prefix_match ===
problem   : a 192-token agent context arriving at a cache of 64 pages
cache     : the previous turn, diverging at page 21 of 24 (one token changed)
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 token compare, 1 candidate            -        -          -         -  0.00e+00  ok    exact, but one sequence only
2 chained id, linear scan               -        -          -         -  0.00e+00  ok    every cached sequence
3 hash probe, serial early exit          -        -          -         -  0.00e+00  ok    O(1) per page
4 hash probe, all pages at once          -        -          -         -  0.00e+00  ok    one probe of depth

Each variant answered three scenarios,

0

Two numbers from that output are worth carrying.

**The lookup is free relative to what it decides.** It reads a few kilobytes of block ids;
charged a generous 20 µs for launches and dependent misses, it is still hundreds of times
cheaper than the prefill it can skip. There is no other kernel in this repository with that
ratio, and it is the reason prefix caching outranks everything else in Part 7's ladder.

**One edited token near the top costs the entire context.** The table showing where an edit
lands is the whole argument for prompt layout: a timestamp, a session id or a randomized
greeting at the start of a system prompt invalidates every page after it. The same text at the
end costs one page. That makes *where you put things in the prompt* a performance decision, and
it is a decision no kernel can rescue you from.

## Part 2 · Fan-out, and the same identity again

Prefix *caching* stops you recomputing the shared prefix. It does not stop you **re-reading**
it, and at decode time reading is the whole cost.

When an agent fans out — a planner spawning sub-agents, a search sampling N continuations, a
retry — you get N sequences that are identical for their first `P` tokens. Run them
independently and every decode step streams that prefix N times:

```
independent:  N × (P + S) × D × 2 × bytes
cascade:      P × D × 2 × bytes   +   N × S × D × 2 × bytes
```

The fix is the online-softmax merge from
[`06_flash_decode.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/06_flash_decode.cu),
pointed at a different partition. Attend to the shared prefix once, producing a partial
`(m, ℓ, acc)` per query; attend to each suffix separately; merge:

$$\ell = \ell_1 e^{m_1-m} + \ell_2 e^{m_2-m} \qquad \text{acc} = \text{acc}_1 e^{m_1-m} + \text{acc}_2 e^{m_2-m}$$

It is character-for-character the identity that merges split-K partials. **Same algebra,
different reason for splitting** — there, to fill idle SMs; here, because a prefix is shared
and a suffix is not.

That is the general lesson: once you have a merge that lets you compute attention over any
partition of the keys and combine the pieces afterwards, "these N requests share a prefix"
stops being a scheduling curiosity and becomes a partition you can exploit in the kernel.

In [5]:
sh("make --no-print-directory 13_prefix_attention")

=== 13_prefix_attention ===
problem   : 4 sequences sharing a 128-token prefix, 32-token suffixes each, head dim 32
scenario  : an agent fan-out — same system prompt, history and tool results,
            different branch. Or one agent's N sampled continuations.
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 independent per sequence              -        -          -         -  5.06e-06  ok    prefix read N times
2 cascade (prefix read once)            -        -          -         -  6.27e-06  ok    prefix read once

Both compute the same function — checked against plain attention over the
concatenated [prefix ; suffix] in double, so the partition-and-merge is
verified rather than assumed

0

Both variants are checked against plain attention over the concatenated `[prefix ; suffix]` in
double precision, so the partition-and-merge is verified rather than assumed. The saving tends
to `(P+S)/S` as the fan-out grows — the prefix becomes free, and only the divergent suffix
costs anything.

And note how much larger that ratio is for agents than for chat. A chat turn's shared prefix
is a system prompt — maybe 200 tokens against 4k of conversation. An agent's shared prefix is
the system prompt **and** the tool definitions **and** every tool result so far, which is
usually most of the context.

## Part 3 · The KV cache held hostage

This is the one that has no kernel, and it is probably the biggest.

While a tool runs — an API call, a search, a code execution — the sequence produces nothing.
It is not decoding. It is not prefilling. It is **holding its entire KV cache** and waiting.
With 2-second tool calls and a couple of hundred milliseconds of generation per turn, the
sequence is idle for most of its life, and its memory is unavailable to anyone else the whole
time.

There are only three things a scheduler can do, and they trade against each other:

| | cost | when it wins |
|---|---|---|
| **hold** | KV occupied for the whole tool call | short tools, spare memory |
| **evict and recompute** | re-prefill the context on return | long tools, expensive memory |
| **offload to host RAM** | a PCIe round trip each way | medium tools, PCIe to spare |

The crossover is arithmetic, and the cell below does it.

In [6]:
# Hold, evict, or offload? Computed for one agent's context.
def tool_gap_policy(context_tokens=20000, kv_bytes_per_token=131072.0, tool_latency_s=2.0,
                    prefill_tok_per_s=20000.0, pcie_gb_per_s=25.0, pool_gb=40.0):
    kv_gb = context_tokens * kv_bytes_per_token / 1e9
    return dict(
        kv_gb=kv_gb,
        # Holding costs nothing in time and everything in memory: the slot is unavailable for
        # the whole gap, so express it as GB-seconds of pool tied up.
        hold_gb_seconds=kv_gb * tool_latency_s,
        # Evicting frees the memory immediately and costs a full re-prefill on return.
        evict_extra_s=context_tokens / prefill_tok_per_s,
        # Offloading frees device memory for the gap, at two PCIe transfers.
        offload_extra_s=2.0 * kv_gb / pcie_gb_per_s,
        # How much of the gap the offload round trip eats — if it exceeds the gap, offloading
        # cannot even finish before the tool returns.
        pool_gb=pool_gb,
    )

# Hold costs no time and all the memory, so it is never the loser on a time axis. The honest
# comparison is: GIVEN that you need the slot back, is it cheaper to recompute or to move?
GAP = 2.0
print(f"{'context':>10}{'KV held':>10}{'hold':>16}{'evict':>12}{'offload':>12}"
      f"{'freeing it':>14}   fits in a 2s gap?")
print("-" * 92)
for ctx in (4000, 10000, 20000, 50000, 100000, 200000):
    p = tool_gap_policy(context_tokens=ctx, tool_latency_s=GAP)
    cheaper = "evict" if p["evict_extra_s"] < p["offload_extra_s"] else "offload"
    fits = "yes" if p["offload_extra_s"] < GAP else f"NO — {p['offload_extra_s']:.1f}s > {GAP}s"
    print(f"{ctx:>10,}{p['kv_gb']:>8.2f} GB{p['hold_gb_seconds']:>13.1f} GB·s"
          f"{p['evict_extra_s']:>10.2f} s{p['offload_extra_s']:>10.2f} s{cheaper:>14}   {fits}")

print(f"""
The `hold` column is not in seconds, and that is the point: holding costs no time at all, so on
a time axis it always "wins". Its cost is GB-seconds of pool tied up, which is a cost you only
feel when someone else wants the memory. So the table asks the question that actually has an
answer — *given* that you need the slot back, is it cheaper to recompute the KV or to move it?

Offloading moves KV over PCIe at ~25 GB/s: {131072/25e9*1e6:.1f} us per token each way,
{2*131072/25e9*1e6:.1f} us round trip. Re-prefilling regenerates it at ~20k tokens/s, which is
{1/20000*1e6:.0f} us per token. Moving a token of KV is roughly {(1/20000)/(2*131072/25e9):.0f}x
cheaper than recomputing it, so offload wins at every context length here — which is why "CPU
offload" appears in every serving engine, and why it is usually the right answer for an agent's
tool gap.

Two things break that in practice. The last column is the first: at 200k context the round trip
no longer fits inside a 2-second gap, so the sequence is still coming back when the tool
returns and you have converted a memory problem into a latency problem. The second is PCIe
*contention* — the same bus is carrying weights, new requests and other sequences' offloads, so
an offload that is free in isolation queues behind everything else at scale.

And there is a fourth option, which costs nothing at all: do not hold the slot. Finish the
turn, drop the sequence entirely, and let the prefix cache make the return cheap. That is what
makes Part 1's hit rate the load-bearing number, and it is why agent serving and prefix caching
are the same conversation.""")

   context   KV held            hold       evict     offload    freeing it   fits in a 2s gap?
--------------------------------------------------------------------------------------------
     4,000    0.52 GB          1.0 GB·s      0.20 s      0.04 s       offload   yes
    10,000    1.31 GB          2.6 GB·s      0.50 s      0.10 s       offload   yes
    20,000    2.62 GB          5.2 GB·s      1.00 s      0.21 s       offload   yes
    50,000    6.55 GB         13.1 GB·s      2.50 s      0.52 s       offload   yes
   100,000   13.11 GB         26.2 GB·s      5.00 s      1.05 s       offload   yes
   200,000   26.21 GB         52.4 GB·s     10.00 s      2.10 s       offload   NO — 2.1s > 2.0s

The `hold` column is not in seconds, and that is the point: holding costs no time at all, so on
a time axis it always "wins". Its cost is GB-seconds of pool tied up, which is a cost you only
feel when someone else wants the memory. So the table asks the question that actually has an
answer — *

In [7]:
# Slot utilization: what fraction of a held slot's life is the GPU doing anything for it.
print(f"{'tool latency':>14}{'turns':>8}{'busy':>10}{'idle':>10}{'slot util':>12}"
      f"{'agents per 40 GB':>19}")
print("-" * 76)
for tool_s in (0.2, 0.5, 1.0, 2.0, 5.0, 15.0):
    a = agent_run(tool_latency_s=tool_s)
    print(f"{tool_s:>12.1f}s{20:>8}{a['prefill_s']+a['decode_s']:>9.1f}s"
          f"{a['tool_s']:>9.1f}s{a['slot_util']:>11.0%}{a['concurrent_agents']:>18.1f}")

print("""
At a 2-second tool call, an agent's slot is doing useful GPU work about a third of the time.
The other two thirds it is a KV cache with a reservation.

That number is the reason agent serving looks so different from chat serving. In chat, a slot
that is not decoding is a slot you can give away. In an agent workload, most slots are not
decoding most of the time — so the scheduler's job stops being "keep the batch full" and
becomes "do not let idle sequences eat the pool". Continuous batching does not help here,
because the sequence has not finished; it is simply not asking for anything.""")

  tool latency   turns      busy      idle   slot util   agents per 40 GB
----------------------------------------------------------------------------
         0.2s      20     51.7s      4.0s        93%              14.0
         0.5s      20     51.7s     10.0s        84%              14.0
         1.0s      20     51.7s     20.0s        72%              14.0
         2.0s      20     51.7s     40.0s        56%              14.0
         5.0s      20     51.7s    100.0s        34%              14.0
        15.0s      20     51.7s    300.0s        15%              14.0

At a 2-second tool call, an agent's slot is doing useful GPU work about a third of the time.
The other two thirds it is a KV cache with a reservation.

That number is the reason agent serving looks so different from chat serving. In chat, a slot
that is not decoding is a slot you can give away. In an agent workload, most slots are not
decoding most of the time — so the scheduler's job stops being "keep the batch full" 

## Part 4 · Every agent is on a different turn

Part 3 was about one sequence's idle time. This is about what the *batch* looks like when you
put many of them together, and it is the place where an agent workload differs from chat in a
way that is easy to miss.

A chat batch is roughly uniform — everyone is a few thousand tokens into a conversation. An
agent batch is a mix of **turn numbers**: one run is on turn 2 with 4k of context, another is on
turn 40 with 60k, having accumulated thirty-eight tool results. Same workload, different points
in its life, same batch.

The textbook batched-attention kernel gives every sequence the same loop bound and masks the
overhang, so its work is `B × Lmax`. The fix is `cu_seqlens` — a prefix sum of the lengths, so
each sequence loops to its own length and the KV buffer is packed.

But that fixes only half of it, and the halves are worth separating carefully:

- **ragged fixes total work.** No more arithmetic on masked-out garbage.
- **splitting fixes the critical path.** With one block per sequence, the 60k sequence still
  runs 15× longer than everyone else, and the step is not over until it finishes. Most of the
  GPU waits for one block. Cutting every sequence into a fixed *number* of chunks — not a fixed
  chunk size — equalizes the blocks, and the pieces merge with the same online-softmax identity
  from Part 2.

A batch can have zero wasted work and still spend most of its time with one SM busy.

In [8]:
sh("make --no-print-directory 17_ragged_batch")

=== 17_ragged_batch ===
problem   : 6 agent sequences in one decode step, 8 to 96 tokens
spread    : longest / shortest = 12.0x — a mix of turn numbers, not a mix of
            users
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 padded to the longest                 -        -          -         -  1.17e-05  ok    reads B x Lmax
2 ragged (cu_seqlens)                   -        -          -         -  1.17e-05  ok    reads only what exists
3 ragged + split, balanced              -        -          -         -  2.55e-05  ok    same bytes, shorter critical path

All three match a per-sequence reference computed in double, so the packing,
the masking and the split-and-merge are each verifie

0

Two things in that output are worth reading against each other.

The **work** table shows padding throwing away a large fraction of the reads. The **critical
path** table shows rows 1 and 2 identical — going ragged did nothing for the imbalance. They are
different problems with different fixes, and conflating them is how a team ships `cu_seqlens`,
measures no latency improvement, and concludes the optimization does not work.

And note what the last table refuses to claim. Padding waste as a *percentage* lands between a
third and a half for any spread wide enough to matter — chat included. What separates an agent
batch is the absolute size: the same 45% is 45% of 4k tokens in one case and of 40k in another.
A table of ratios would have said the opposite of the truth, which is worth remembering the next
time a ratio is the headline.

## Part 5 · Constrained decoding, and a lesson in what not to optimize

An agent's output is a tool call: JSON, fixed schema, known function names. Serving stacks
enforce that with a grammar — at each step the parser says which tokens are legal and
everything else is masked to `-inf`.

That mask is one float per vocabulary entry per sequence, and vocabularies are large. A dense
fp32 mask at 128k vocab and batch 32 is **16.4 MB read per decode step**.

Which sounds alarming and is the wrong thing to be alarmed about — and that is why this kernel
is in the notebook.

In [9]:
sh("make --no-print-directory 14_logit_mask")

=== 14_logit_mask ===
problem   : vocab 4096, batch 4, 2 sequences on a tight grammar and 2 on a loose one
allowed   : 42.3% of the vocabulary on average
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 dense fp32 mask                       -        -          -         -  1.11e-07  ok    mask = 4 B/token
2 bitset mask                           -        -          -         -  1.11e-07  ok    mask = 1 bit/token
3 token ranges                          -        -          -         -  1.11e-07  ok    touches allowed only
4 ranges, greedy (temp 0)               -        -          -         -  1.11e-07  ok    no softmax pass

All four pick the same token — the check requires exact agreement on 

0

**The mask is 0.05% of the step.** Five microseconds out of ten milliseconds. The kernel your
instinct says to optimize is not where the time is.

Three things survive that:

1. **The bitset is free.** Same semantics, 1/32 the bytes. There is no trade to weigh.
2. **The fraction is not always 0.05%.** Speculative decoding samples `k+1` positions per
   step; a draft model has a fraction of the weights and the same vocabulary; batch scales the
   mask but not the weight read. Push those together and the sampling step stops being noise.
3. **The real cost is not on the GPU.** Advancing a grammar and computing the allowed set is
   CPU work, per sequence, per step. Unoverlapped, that is *milliseconds* — a thousand times
   the number this kernel is about. See
   [Structured Output & Guided Decoding](./Structured_Output_Guided_Decoding.ipynb).

The habit worth taking from this: divide by the step before optimizing anything. It is the
same discipline as [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb), applied
one level up.

## Part 6 · The one place agents get a speedup nobody else gets

Part 5 ended by dividing the mask by the step and finding 0.05%. This part changes the
denominator, and it is the most agent-specific result in the notebook.

Speculative decoding: a small draft model proposes `k` tokens, the big model scores all `k+1`
positions in **one** forward pass, and verification keeps the longest leading run the big model
agrees with. One weight read, up to `k+1` tokens out. The expected tokens per weight read is

$$E = \frac{1 - \alpha^{k+1}}{1 - \alpha}$$

where α is the per-token acceptance rate. Note that α enters as `α^(k+1)` — a small change in
acceptance is a large change in how long a draft is worth running.

Which is why agents are the workload speculation was waiting for. An agent's output is a **tool
call**: `{"name": "search", "arguments": {"query": ` is fixed by the schema before the model has
decided anything. A tiny draft model, a lookup table, or the grammar itself gets those right
almost every time, and agents run at temperature 0 anyway, so greedy verification — the
cheapest and strictest form — is exactly what is wanted. Acceptance rates that would be
optimistic for prose are conservative here.

The verification itself has a small lesson of its own. The textbook description ("walk the
positions, stop at the first mismatch") turns directly into a serial chain of dependent argmaxes
over a 128k vocabulary, all of it on the critical path of the step it is meant to accelerate.
The positions are independent — the big model already produced every logit row in one pass — so
computing them all and reducing to the first mismatch does more total work and finishes sooner.

In [10]:
sh("make --no-print-directory 18_spec_verify")

g++ -std=c++17 -O2 -I. -I../kernelbench/shim -pthread -Wno-unknown-pragmas -x c++ 18_spec_verify.cu -o build/cpu/18_spec_verify
=== 18_spec_verify ===
problem   : a 4-token draft verified against the target's 5 logit rows
grammar   : 3.9% of the 1024-token vocabulary legal on average
outcome   : 2 of 4 draft tokens accepted, plus the bonus token
ties      : every row has two bit-identical maxima, so the lower-index
            tie-break is exercised rather than assumed
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 serial, stop at first reject          -        -          -         -  0.00e+00  ok    POS dependent argmaxes
2 all positions at once                 -        -          -      

0

The second table is the one to sit with, because it revises Part 5's conclusion without
contradicting it.

The mask ran once per weight read. Under speculation it runs `k+1` times against the *same*
weight read, and if the draft model is doing the drafting then those runs are against an eighth
of the weights. 0.05% becomes 0.3%, then 0.5%. Still small — but now the same order as things
teams do spend a sprint on, and the bitset takes it back to nothing for free.

**"What fraction of the step is this" has no fixed answer.** It is a ratio, and speculation
changes the denominator. That is not a correction to Part 5; it is the reason Part 5's habit —
divide by the step — has to be repeated whenever the step changes, rather than done once and
written down.

## Part 7 · Forking a trajectory

A planner spawns N sub-agents from the same state. A search samples N continuations. A retry
re-runs from the same context. In every case N sequences begin identical and diverge.

The naive implementation copies the parent's KV cache N times. PagedAttention already has the
answer, because it already has the indirection: a sequence's KV is a **block table**, so
forking copies a list of integers and shares the pages, with a reference count.

One page is the exception, and it is the whole of copy-on-write: the parent's *last* page is
partially filled and each child will append into it, so that one page is copied per child.
Everything before it is immutable — those tokens are in the past and no branch will ever write
to them again.

In [11]:
sh("make --no-print-directory 15_kv_fork")

=== 15_kv_fork ===
problem   : parent of 37 tokens forks into 4 children, each generating 5 more
paging    : 8 tokens/page, parent occupies 5 pages, last one 5/8 full
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 deep copy per child                   -        -          -         -  4.69e-06  ok    full duplication, 29 pages
2 copy-on-write tail only               -        -          -         -  4.69e-06  ok    shares full pages, 13 pages

Both produce the same attention output for every child — checked against a
reference over [parent ; child] in double. Sharing pages between branches is
invisible to the read path, which is exactly why it is safe.

What the fork cost:

  strategy       

0

Both layouts produce identical attention output for every child, checked against a reference
over `[parent ; child]` in double — sharing pages between branches is invisible to the read
path, which is exactly why it is safe.

The scaling is the point: copy-on-write is **O(1) in the parent's length**. One partial page
per child, however long the conversation. That is what makes deep agent trees — fork, explore,
fork again — affordable, and it is the same indirection PagedAttention introduced for a
completely different reason.

## Part 8 · Why determinism decides your cache hit rate

This one connects back to [Training Kernels & Memory](./Training_Kernels_And_Memory.ipynb),
and it is the least obvious link in the notebook.

An agent replays. It retries failed tool calls, resumes interrupted runs, re-runs the same
trajectory in an eval harness, and caches results keyed on the conversation so far. Every one
of those depends on the same input producing the same output.

But a decode step is not automatically deterministic. Reductions that accumulate with
`atomicAdd` commit in whatever order blocks finish, floating-point addition is not
associative, and the result differs in the last bits. Usually that changes nothing. Sometimes
it flips a token — and once one token differs, **the rest of the trajectory is a different
conversation**, the prefix cache misses from that point on, and the replay is not a replay.

The same mechanism produces batch-dependence: a request's logits can depend on what else was
in its batch, because the batch composition changes the reduction shape.
`07_rmsnorm_backward.cu` demonstrates the effect directly, and the eval harness turns it into
a gate.

In [12]:
# The same binary, four different block orders. This is GPU non-determinism reproduced on a
# CPU: kernelbench's shim permutes block order under KB_SHIM_SHUFFLE, so an order-dependent
# reduction gives a different answer with a different seed — deterministically, in a second.
import subprocess
BIN = KERNELS / "build" / "cpu" / "07_rmsnorm_backward"
if not BIN.exists():
    sh("make --no-print-directory HAVE_NVCC=no build/cpu/07_rmsnorm_backward")

def checksums(seed=None):
    env = dict(os.environ)
    if seed is not None:
        env["KB_SHIM_SHUFFLE"] = str(seed)
    out = subprocess.run([str(BIN)], capture_output=True, text=True, env=env).stdout
    line = next(l for l in out.splitlines() if l.startswith("##KB##"))
    return {v["name"]: v["checksum"][:10] for v in json.loads(line[7:])["variants"]}

base = checksums()
seen = {n: {c} for n, c in base.items()}
for seed in (1, 5, 42, 1337):
    for n, c in checksums(seed).items():
        seen[n].add(c)

print(f"{'reduction':<34}{'distinct answers':>18}   verdict")
print("-" * 76)
for n, s in seen.items():
    print(f"{n:<34}{len(s):>18}   "
          f"{'ORDER-DEPENDENT' if len(s) > 1 else 'reproducible'}")

print("""
The atomic version is not wrong — every ordering is a valid answer, and they differ at the
level of fp32 rounding. It is just not *reproducible*, and for an agent that is a different
property with different consequences: a replay that diverges is a cache miss, a failed eval
comparison, and a bug you cannot bisect.

Which is why serving stacks that promise reproducible output use fixed-split reductions rather
than atomics, and why "batch-invariant kernels" became a topic at all. It costs one extra
kernel launch and some scratch. For an agent platform it buys you the ability to replay.""")

reduction                           distinct answers   verdict
----------------------------------------------------------------------------
1 atomic dW (not reproducible)                     4   ORDER-DEPENDENT
2 deterministic dW                                 1   reproducible
3 + float4, x in registers                         1   reproducible
4 + rstd saved from forward                        1   reproducible

The atomic version is not wrong — every ordering is a valid answer, and they differ at the
level of fp32 rounding. It is just not *reproducible*, and for an agent that is a different
property with different consequences: a replay that diverges is a cache miss, a failed eval
comparison, and a bug you cannot bisect.

Which is why serving stacks that promise reproducible output use fixed-split reductions rather
than atomics, and why "batch-invariant kernels" became a topic at all. It costs one extra
kernel launch and some scratch. For an agent platform it buys you the ability to r

### The failure that survives a fixed random seed

`07`'s atomic reduction is non-deterministic in an obvious way: blocks commit in whatever order
they finish. Fix the order and the problem goes away.

There is a second failure that a fixed seed, a fixed input and a thousand identical reruns will
never surface. Consider the reduction behind a softmax denominator or an LM-head logit. A
throughput-minded kernel picks its split count from the **batch**: one sequence in flight means
idle SMs, so cut the row into many chunks; sixty-four sequences means the machine is full, so
use one. Both are correct, nothing races, and every run at a fixed batch size gives the same
answer.

But the *shape* of the reduction tree just changed with the batch size, and floating-point
addition is not associative. The request did not change. Its neighbours did.

In [13]:
sh("make --no-print-directory 19_batch_invariant")

g++ -std=c++17 -O2 -I. -I../kernelbench/shim -pthread -Wno-unknown-pragmas -x c++ 19_batch_invariant.cu -o build/cpu/19_batch_invariant
=== 19_batch_invariant ===
problem   : sum a 512-element row — a softmax denominator, a LayerNorm mean, a
            logit against the LM head
range     : mixed magnitudes — 1 element in 64 is ~273067x the others, which is
            what makes summation order visible at all
device    : CPU (cuda_shim emulation)
peak      : n/a — this is the CPU emulation, correctness only.
            Timings are omitted rather than reported as GPU numbers.

variant                         median ms    ±MAD       GB/s   GFLOP/s   max err  
--------------------------------------------------------------------------------------------
1 split chosen from batch size          -        -          -         -  2.33e-07  ok    fast, NOT batch-invariant
2 fixed split, always 8                 -        -          -         -  1.16e-07  ok    batch-invariant, and checked
3 one 

0

The adaptive kernel gave several different answers to the same question at different batch
sizes. The fixed-split kernel gave one, bit for bit.

Three things follow.

1. **This is not a bug you can test for without varying the batch.** Rerunning the same request
   a thousand times at batch 1 proves nothing. Neither does `KB_SHIM_SHUFFLE`, which permutes
   block *order* — here the order is already fixed and it is the *shape* that moves.
2. **Kahan summation does not fix it.** It is in the table because it is what people reach for.
   It reduces the error; it does not make the answer independent of the tree, because the tree
   still changes.
3. **The cost is real and it is small.** Fixing the split loses the small-batch latency win,
   where the adaptive kernel would have used idle SMs. For a chat product that never replays
   anything, that is a bad trade. For an agent platform running evals, retries and a prefix
   cache keyed on its own output, it is an easy one.

## Part 9 · Putting it together

Five levers, applied to the same 40-turn agent, in the order a serving team would reach for
them.

In [14]:
BASE = dict(turns=40, system_tokens=2000, tool_def_tokens=1500, user_tokens=200,
            response_tokens=150, tool_result_tokens=2000, tool_latency_s=2.0,
            fanout=1, prefix_hit_rate=0.0)

LEVERS = [
    ("no caching at all",              dict()),
    ("+ prefix cache at 80%",          dict(prefix_hit_rate=0.80)),
    ("+ tune it to 99%",               dict(prefix_hit_rate=0.99)),
    ("+ shorter tool results (2k→800)", dict(prefix_hit_rate=0.99, tool_result_tokens=800)),
    ("+ overlap tools (2s→0.5s eff.)", dict(prefix_hit_rate=0.99, tool_result_tokens=800,
                                            tool_latency_s=0.5)),
]

print(f"{'configuration':<34}{'prefill tok':>13}{'wall':>9}{'$':>9}{'slot util':>11}"
      f"{'agents/40GB':>13}")
print("-" * 90)
prev = None
for name, over in LEVERS:
    a = agent_run(**{**BASE, **over})
    delta = "" if prev is None else f"  ({100*(a['wall_s']/prev-1):+.0f}% wall)"
    print(f"{name:<34}{a['prefill_actual']:>13,.0f}{a['wall_s']:>8.0f}s"
          f"{a['cost_usd']:>9.2f}{a['slot_util']:>10.0%}{a['concurrent_agents']:>13.1f}{delta}")
    prev = a["wall_s"]

print("""
The ordering is the lesson. Prefix caching is worth more than everything else combined, and it
is not a kernel — it is a hash lookup and a decision not to recompute. The kernels in this
notebook matter *after* that: cascade attention for the fan-out, copy-on-write for the fork,
a bitset for the mask.

Watch the slot-utilization column fall as the configuration improves. That is not a regression:
the GPU work shrank while the tool gap did not, so a *smaller* fraction of the slot's life is
GPU time. Utilization is a diagnostic here, not a goal — chasing it upward would mean doing
more GPU work, which is the opposite of the exercise. Only the last row raises it honestly, by
shrinking the idle side.

And that last row is the one that is not about the GPU at all. Overlapping tool calls — issuing
them concurrently, or speculatively starting the next turn — moves the largest single term in
the wall clock, and no kernel will ever touch it.""")

configuration                       prefill tok     wall        $  slot util  agents/40GB
------------------------------------------------------------------------------------------
no caching at all                     1,825,000     271s     5.56       71%          3.5
+ prefix cache at 80%                   435,040     202s     1.40       60%          3.5  (-26% wall)
+ tune it to 99%                        104,924     185s     0.40       57%          3.5  (-8% wall)
+ shorter tool results (2k→800)          49,232     182s     0.24       56%          7.5  (-2% wall)
+ overlap tools (2s→0.5s eff.)           49,232     122s     0.24       84%          7.5  (-33% wall)

The ordering is the lesson. Prefix caching is worth more than everything else combined, and it
is not a kernel — it is a hash lookup and a decision not to recompute. The kernels in this
notebook matter *after* that: cascade attention for the fan-out, copy-on-write for the fork,
a bitset for the mask.

Watch the slot-utili

## What to take away

1. **Agent prefill is quadratic in turns** unless the prefix cache stops it. That is the
   dominant cost, it is not really a kernel problem, and a 95% hit rate is meaningfully worse
   than 99% because the misses are misses on the *whole context*. The one kernel that does
   touch it — the block-hash lookup in `16` — has a better cost-to-value ratio than anything
   else in this repository, and its main lesson is about prompt *layout*: an edited token near
   the top of a system prompt invalidates every page after it.
2. **The online-softmax merge generalizes, twice over.** It was invented to split one KV cache
   across idle SMs. The same identity lets N branches share a prefix read (`13`), and lets a
   ragged batch be cut into equal pieces so no sequence sets the step time alone (`17`). Look
   for it whenever a set of sequences shares a partition of the keys.
3. **Idle slots are the scheduler's real problem.** An agent holding its KV through a
   two-second tool call is using a third of its slot. Hold, evict or offload is arithmetic —
   and the fourth option, dropping the sequence and relying on the prefix cache, is usually
   the best one.
4. **Ragged and balanced are different fixes.** `cu_seqlens` removes wasted work; splitting
   removes the imbalance. A batch can have neither wasted work nor any parallelism, and
   shipping the first fix while measuring the second is how an optimization gets written off.
5. **Divide by the step — and then do it again when the step changes.** The logit mask looks
   alarming at 16 MB and is 0.05% of a decode step. Under speculation against a small draft
   model it is 0.5%. Neither number is wrong; the ratio simply has a denominator, and
   speculation moves it.
6. **Agents are the workload speculation was waiting for.** Acceptance enters as `α^(k+1)`, and
   a schema-constrained tool call is the most predictable text a model ever emits. The same
   draft depth that buys 2.3× on prose buys 4× here.
7. **Determinism is an agent feature, and it has two independent failure modes.** Atomic
   reductions vary with block *order* — `KB_SHIM_SHUFFLE` finds those. Adaptive split counts
   vary with batch *shape* — nothing finds those except varying the batch, which is why `19`
   exists. Both end the same way: one flipped token, and the rest of the trajectory is a
   different conversation.

### Next

- [RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) — prompt layout, the
  cache hierarchy, and semantic caching's sharp edge.
- [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) — where the acceptance
  formula in Part 6 comes from, and the regime where speculation makes you slower.
- [Structured Output & Guided Decoding](./Structured_Output_Guided_Decoding.ipynb) — the CPU
  side of Part 4, which is the part that actually costs milliseconds.
- [Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) — where the merge
  identity in Part 2 comes from.
- [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) — the machine
  all of this runs on.